In [1]:
import os
import cv2
import math
import torch
import random
import warnings
import numpy as np
import pandas as pd

from glob import glob
from tqdm import tqdm
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split

import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import mediapipe as mp

warnings.filterwarnings("ignore")

2026-05-18 12:28:34.137126: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-18 12:28:34.139671: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-18 12:28:34.185339: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-18 12:28:35.058807: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
class CFG:

    # ───────────────── DEVICE ─────────────────
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # ───────────────── DATA ─────────────────
    IMG_SIZE = 128
    FPS = 30.0
    SEQ_LEN = 512
    MIN_FRAMES = 64

    # ───────────────── MODEL ─────────────────
    D_MODEL = 128
    NHEAD = 8
    NUM_LAYERS = 4
    DIM_FF = 512
    DROPOUT = 0.1

    # ───────────────── TRAINING ─────────────────
    BATCH_SIZE = 2 if torch.cuda.is_available() else 1
    EPOCHS = 100 if torch.cuda.is_available() else 10
    LR = 5e-5
    NUM_WORKERS = 0

    # ───────────────── DATASET ─────────────────
    UBFC_ROOT = "/home/dhruv/Documents/rppg project/UBFC/dataset 2"
    GT_FILENAME = "ground_truth.txt"

    # ───────────────── CHECKPOINTS ─────────────────
    SAVE_BEST = "best_rppg_ubfc.pth"
    SAVE_FINAL = "final_rppg_ubfc.pth"

cfg = CFG()

print("Device:", cfg.DEVICE)

Device: cpu


In [3]:
mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

In [5]:
def temporal_difference(frames):

    diff_frames = []

    for i in range(1, len(frames)):

        prev = frames[i - 1].astype(np.float32)
        curr = frames[i].astype(np.float32)

        diff = (curr - prev) / (curr + prev + 1e-8)

        diff_frames.append(diff)

    return np.array(diff_frames, dtype=np.float32)

In [6]:
def POS_WANG(rgb_signals):

    rgb = np.asarray(rgb_signals, dtype=np.float32)

    mean = np.mean(rgb, axis=0)

    rgb = rgb / (mean + 1e-8)

    projection = np.array([
        [0, 1, -1],
        [-2, 1, 1]
    ], dtype=np.float32)

    S = projection @ rgb.T

    alpha = np.std(S[0]) / (np.std(S[1]) + 1e-8)

    h = S[0] + alpha * S[1]

    return h

In [7]:
def bandpass_filter(signal, fs=30.0):

    lowcut = 0.75
    highcut = 3.0

    nyquist = fs * 0.5

    b, a = butter(
        5,
        [lowcut / nyquist, highcut / nyquist],
        btype='band'
    )

    min_len = 3 * max(len(a), len(b))

    if len(signal) < min_len:

        sig = signal - signal.mean()

        return sig / (sig.std() + 1e-8)

    filtered = filtfilt(b, a, signal)

    filtered = (
        filtered - filtered.mean()
    ) / (
        filtered.std() + 1e-8
    )

    return filtered.astype(np.float32)

In [8]:
def compute_bpm(signal, fs=30.0):

    signal = bandpass_filter(signal, fs)

    n = len(signal)

    freqs = np.fft.rfftfreq(
        n,
        d=1.0 / fs
    )

    fft_mag = np.abs(np.fft.rfft(signal))

    mask = (freqs >= 0.75) & (freqs <= 3.0)

    freqs_band = freqs[mask]
    fft_band = fft_mag[mask]

    peak_idx = np.argmax(fft_band)

    # quadratic interpolation
    if 0 < peak_idx < len(fft_band) - 1:

        alpha = fft_band[peak_idx - 1]
        beta = fft_band[peak_idx]
        gamma = fft_band[peak_idx + 1]

        denominator = alpha - 2 * beta + gamma

        if abs(denominator) > 1e-8:
            p = 0.5 * (alpha - gamma) / denominator
        else:
            p = 0.0

        freq_res = freqs_band[1] - freqs_band[0]

        peak_freq = freqs_band[peak_idx] + p * freq_res

    else:
        peak_freq = freqs_band[peak_idx]

    bpm = peak_freq * 60.0

    return float(bpm)

In [9]:
all_video_paths = sorted(
    glob(os.path.join(cfg.UBFC_ROOT, "subject*", "*.avi"))
)

all_gt_paths = sorted(
    glob(os.path.join(cfg.UBFC_ROOT, "subject*", cfg.GT_FILENAME))
)

print("Videos:", len(all_video_paths))
print("GT:", len(all_gt_paths))

Videos: 31
GT: 31


In [10]:
combined = list(zip(all_video_paths, all_gt_paths))

train_combined, val_combined = train_test_split(
    combined,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_video_paths = [x[0] for x in train_combined]
train_gt_paths = [x[1] for x in train_combined]

val_video_paths = [x[0] for x in val_combined]
val_gt_paths = [x[1] for x in val_combined]

print("Train subjects:", len(train_video_paths))
print("Val subjects:", len(val_video_paths))

Train subjects: 24
Val subjects: 7


In [11]:
class RPPGDataset(Dataset):

    def __init__(self, video_paths, gt_paths):

        self.video_paths = video_paths
        self.gt_paths = gt_paths

    def __len__(self):

        return len(self.video_paths)

    def __getitem__(self, idx):

        video_path = self.video_paths[idx]
        gt_path = self.gt_paths[idx]

        cap = cv2.VideoCapture(video_path)

        raw_frames = []
        rgb_signals = []

        frame_idx = 0

        while True:

            ret, frame = cap.read()

            if not ret:
                break

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            results = face_mesh.process(frame)

            if not results.multi_face_landmarks:
                continue

            h, w, _ = frame.shape

            landmarks = results.multi_face_landmarks[0]

            xs = [lm.x for lm in landmarks.landmark]
            ys = [lm.y for lm in landmarks.landmark]

            x_min = max(int(min(xs) * w), 0)
            x_max = min(int(max(xs) * w), w)

            y_min = max(int(min(ys) * h), 0)
            y_max = min(int(max(ys) * h), h)

            face = frame[y_min:y_max, x_min:x_max]

            if face.size == 0:
                continue

            face = cv2.resize(
                face,
                (cfg.IMG_SIZE, cfg.IMG_SIZE)
            )

            raw_frames.append(face)

            rgb_mean = np.mean(face, axis=(0, 1))

            rgb_signals.append(rgb_mean)

            frame_idx += 1

        cap.release()

        raw_frames = np.array(raw_frames)

        if len(raw_frames) < cfg.MIN_FRAMES:

            dummy = torch.zeros(
                3,
                cfg.SEQ_LEN,
                cfg.IMG_SIZE,
                cfg.IMG_SIZE
            )

            signal = torch.zeros(cfg.SEQ_LEN)

            mask = torch.zeros(cfg.SEQ_LEN)

            return dummy, dummy, signal, mask

        diff_frames = temporal_difference(raw_frames)

        raw_frames = raw_frames[:-1]

        raw_frames = raw_frames[:cfg.SEQ_LEN]
        diff_frames = diff_frames[:cfg.SEQ_LEN]

        gt_signal = np.loadtxt(gt_path)

        gt_signal = gt_signal[:len(raw_frames)]

        gt_signal = bandpass_filter(gt_signal)

        seq_len = len(raw_frames)

        mask = np.zeros(cfg.SEQ_LEN)

        mask[:seq_len] = 1.0

        padded_raw = np.zeros(
            (
                cfg.SEQ_LEN,
                cfg.IMG_SIZE,
                cfg.IMG_SIZE,
                3
            ),
            dtype=np.float32
        )

        padded_diff = np.zeros_like(padded_raw)

        padded_signal = np.zeros(cfg.SEQ_LEN)

        padded_raw[:seq_len] = raw_frames
        padded_diff[:seq_len] = diff_frames
        padded_signal[:seq_len] = gt_signal

        padded_raw = torch.tensor(
            padded_raw
        ).permute(3,0,1,2)

        padded_diff = torch.tensor(
            padded_diff
        ).permute(3,0,1,2)

        padded_signal = torch.tensor(
            padded_signal,
            dtype=torch.float32
        )

        mask = torch.tensor(
            mask,
            dtype=torch.float32
        )

        return padded_diff, padded_raw, padded_signal, mask

In [12]:
train_dataset = RPPGDataset(
    train_video_paths,
    train_gt_paths
)

val_dataset = RPPGDataset(
    val_video_paths,
    val_gt_paths
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS
)

In [13]:
class RPPGTransformer(nn.Module):

    def __init__(self):

        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(6, 32, 3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(128, cfg.D_MODEL)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.D_MODEL,
            nhead=cfg.NHEAD,
            dim_feedforward=cfg.DIM_FF,
            dropout=cfg.DROPOUT,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=cfg.NUM_LAYERS
        )

        self.regressor = nn.Linear(cfg.D_MODEL, 1)

    def forward(
        self,
        diff_frames,
        raw_frames,
        src_key_padding_mask=None
    ):

        B, C, T, H, W = raw_frames.shape

        x = torch.cat(
            [diff_frames, raw_frames],
            dim=1
        )

        x = x.permute(0,2,1,3,4)

        x = x.reshape(B*T, 6, H, W)

        x = self.cnn(x)

        x = x.flatten(1)

        x = self.fc(x)

        x = x.reshape(B, T, cfg.D_MODEL)

        x = self.transformer(
            x,
            src_key_padding_mask=src_key_padding_mask
        )

        x = self.regressor(x)

        x = x.squeeze(-1)

        return x

In [14]:
def negative_pearson(preds, labels, mask=None):

    if mask is not None:

        preds = preds * mask
        labels = labels * mask

    preds_c = preds - preds.mean(dim=1, keepdim=True)
    labels_c = labels - labels.mean(dim=1, keepdim=True)

    numerator = torch.sum(
        preds_c * labels_c,
        dim=1
    )

    denominator = torch.sqrt(
        torch.sum(preds_c**2, dim=1) *
        torch.sum(labels_c**2, dim=1)
    )

    loss = 1.0 - numerator / (denominator + 1e-8)

    return loss.mean()

In [15]:
def enhanced_spectral_loss(
    pred,
    target,
    fs=30.0
):

    pred = (
        pred - pred.mean(dim=1, keepdim=True)
    ) / (
        pred.std(dim=1, keepdim=True) + 1e-8
    )

    target = (
        target - target.mean(dim=1, keepdim=True)
    ) / (
        target.std(dim=1, keepdim=True) + 1e-8
    )

    T = pred.shape[1]

    freqs = torch.fft.rfftfreq(
        T,
        d=1.0/fs
    ).to(pred.device)

    band = (freqs >= 0.75) & (freqs <= 3.0)

    pred_fft = torch.abs(
        torch.fft.rfft(pred, dim=1)
    )[:, band]

    target_fft = torch.abs(
        torch.fft.rfft(target, dim=1)
    )[:, band]

    l1 = torch.mean(
        torch.abs(pred_fft - target_fft)
    )

    pred_norm = pred_fft / (
        pred_fft.sum(dim=1, keepdim=True) + 1e-8
    )

    target_norm = target_fft / (
        target_fft.sum(dim=1, keepdim=True) + 1e-8
    )

    kl = F.kl_div(
        torch.log(pred_norm + 1e-8),
        target_norm,
        reduction='batchmean'
    )

    return l1 + 0.1 * kl

In [16]:
model = RPPGTransformer().to(cfg.DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.LR
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.EPOCHS
)

In [17]:
def mean_absolute_bpm_error(
    preds,
    targets,
    fs=30.0
):

    pred_bpms = []
    gt_bpms = []

    for pred, gt in zip(preds, targets):

        pred_bpm = compute_bpm(pred, fs)
        gt_bpm = compute_bpm(gt, fs)

        pred_bpms.append(pred_bpm)
        gt_bpms.append(gt_bpm)

    pred_bpms = np.array(pred_bpms)
    gt_bpms = np.array(gt_bpms)

    mae = np.mean(
        np.abs(pred_bpms - gt_bpms)
    )

    return mae

In [18]:
def train_one_epoch(loader):

    model.train()

    total_loss = 0.0

    for diff_frames, raw_frames, signal, mask in tqdm(loader):

        diff_frames = diff_frames.to(cfg.DEVICE)
        raw_frames = raw_frames.to(cfg.DEVICE)
        signal = signal.to(cfg.DEVICE)
        mask = mask.to(cfg.DEVICE)

        pad_mask = (mask == 0)

        optimizer.zero_grad()

        pred = model(
            diff_frames,
            raw_frames,
            src_key_padding_mask=pad_mask
        )

        pearson = negative_pearson(
            pred,
            signal,
            mask
        )

        smooth = F.smooth_l1_loss(
            pred * mask,
            signal * mask
        )

        spec = enhanced_spectral_loss(
            pred * mask,
            signal * mask
        )

        loss = (
            0.7 * pearson +
            0.2 * smooth +
            0.1 * spec
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    return total_loss / len(loader)

In [19]:
@torch.no_grad()
def validate(loader):

    model.eval()

    total_loss = 0.0

    all_preds = []
    all_targets = []

    pearson_vals = []
    smooth_vals = []
    spec_vals = []

    for diff_frames, raw_frames, signal, mask in tqdm(loader):

        diff_frames = diff_frames.to(cfg.DEVICE)
        raw_frames = raw_frames.to(cfg.DEVICE)
        signal = signal.to(cfg.DEVICE)
        mask = mask.to(cfg.DEVICE)

        pad_mask = (mask == 0)

        pred = model(
            diff_frames,
            raw_frames,
            src_key_padding_mask=pad_mask
        )

        pearson = negative_pearson(
            pred,
            signal,
            mask
        )

        smooth = F.smooth_l1_loss(
            pred * mask,
            signal * mask
        )

        spec = enhanced_spectral_loss(
            pred * mask,
            signal * mask
        )

        loss = (
            0.7 * pearson +
            0.2 * smooth +
            0.1 * spec
        )

        total_loss += loss.item()

        pearson_vals.append(pearson.item())
        smooth_vals.append(smooth.item())
        spec_vals.append(spec.item())

        all_preds.append(
            pred.cpu().numpy()
        )

        all_targets.append(
            signal.cpu().numpy()
        )

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    mae_bpm = mean_absolute_bpm_error(
        all_preds,
        all_targets,
        fs=cfg.FPS
    )

    print("\nValidation Diagnostics")
    print("Pearson :", np.mean(pearson_vals))
    print("SmoothL1:", np.mean(smooth_vals))
    print("Spectral:", np.mean(spec_vals))

    return total_loss / len(loader), mae_bpm

In [ ]:
best_mae = float("inf")

for epoch in range(cfg.EPOCHS):

    train_loss = train_one_epoch(train_loader)

    val_loss, mae_bpm = validate(val_loader)

    print(
        f"\nEpoch {epoch+1}/{cfg.EPOCHS}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Val Loss: {val_loss:.4f}"
        f" | MAE BPM: {mae_bpm:.2f}"
    )

    if mae_bpm < best_mae:

        best_mae = mae_bpm

        torch.save(
            model.state_dict(),
            cfg.SAVE_BEST
        )

        print(
            f"New best model saved "
            f"→ {cfg.SAVE_BEST}"
        )

torch.save(
    model.state_dict(),
    cfg.SAVE_FINAL
)

print("\nTraining complete.")
print("Best MAE:", best_mae)

  0%|                                                    | 0/24 [00:00<?, ?it/s]